In [2]:
# Parameters
model_name = "mamba"
TICKERS = ["AAPL", "INTC", "LCID"]
SEEDS = [4718]
num_epochs = 5
LEARNING_RATE = 5e-4
PREPROCESSING_CONTRACT_VERSION = 1
MAX_SAMPLES_PER_ORDER = 100
SPLIT_CAP_RANDOM_SEED = 4718
NUM_TIME_STEPS = 30
WANDB_ENABLE = True
WANDB_PROJECT = f"loss-grid-search-static"
WANDB_MODE = "online"
ORDERS_PER_BATCH = 64
BATCH_SIZE = 1024
SIGMA = 0.1

# Optional shard parameters populated by Slurm array jobs.
selected_alpha = None
selected_beta_l3 = None
array_job_id = ""
array_task_id = -1

# Static DeepHit Loss Grid Search (Split-Parquet Pipeline)

This grid-search notebook uses static split parquet inputs (`train`/`val`/`test`) and evaluates with unweighted static metrics while preserving shard output compatibility columns.

In [3]:
import gc
import importlib
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import wandb
from dotenv import find_dotenv, load_dotenv
from sklearn.metrics import auc, precision_recall_curve, roc_auc_score

repo_root = None
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "src").exists():
        repo_root = candidate.resolve()
        if str(repo_root) not in sys.path:
            sys.path.insert(0, str(repo_root))
        break
if repo_root is None:
    raise RuntimeError("Could not locate project root containing 'src'.")

from src.models import (
    DeepHitMambaCompeting,
    DeepHitRNNCompeting,
    DeepHitRNNTransformerCompeting,
    DeepHitTransformerCompeting,
)
from src.notebook_data import (
    LabTransform,
    apply_dynamic_normalizer,
    build_order_batch_indices,
    extract_lob_features,
    extract_toxicity_features,
    recensor_after_horizon,
)
from src.notebook_evaluation import (
    standard_brier_score,
    uninformed_brier_score,
    uninformed_cif_curve_from_train,
)
from src.notebook_losses import static_deephit_total_loss

DATASET_BASE_DIR = Path("/ocean/projects/cis260122p/shared/data/datasets")
DATASET_STEM_TEMPLATE = "labeled_dataset_XNAS_ITCH_{ticker}_mbo_20251001_20260101"

REQUIRED_META_KEYS = ["t_max"]
REQUIRED_SPLIT_COLUMNS = {
    "entry_time",
    "duration_s",
    "event_type",
    "side",
    "entry_representation",
    "toxicity_representation",
}

EVENT_NAMES = ["FAVORABLE_FILL", "TOXIC_FILL"]
EVENT_CODES = [1, 2]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
AMP_DTYPE = torch.float16
LOADER_PIN_MEMORY = device.type != "cpu"
AUTOCAST_DEVICE = "cuda" if device.type == "cuda" else "cpu"

print("Device:", device)
print("PyTorch:", torch.__version__)
print("Python:", sys.executable)

if str(model_name).lower() == "mamba":
    def _safe_import_status(module_name: str) -> tuple[bool, str | None, str | None]:
        try:
            module = importlib.import_module(module_name)
            return True, str(getattr(module, "__version__", "unknown")), None
        except Exception as exc:
            return False, None, f"{type(exc).__name__}: {exc}"

    causal_ok, causal_ver, causal_err = _safe_import_status("causal_conv1d")
    mamba_ok, mamba_ver, mamba_err = _safe_import_status("mamba_ssm")
    print("causal_conv1d:", causal_ver if causal_ok else f"broken ({causal_err})")
    print("mamba_ssm:", mamba_ver if mamba_ok else f"broken ({mamba_err})")

dotenv_path = find_dotenv(filename=".env", usecwd=True)
if dotenv_path:
    load_dotenv(dotenv_path=dotenv_path, override=False)
else:
    load_dotenv(override=False)

WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "").strip()
WANDB_API_KEY = os.environ.get("WANDB_API_KEY", "").strip()
if WANDB_ENABLE and WANDB_API_KEY:
    os.environ["WANDB_API_KEY"] = WANDB_API_KEY
    wandb.login(key=WANDB_API_KEY, relogin=True)


def _ticker_paths(ticker_symbol: str) -> dict[str, Path]:
    symbol = str(ticker_symbol).upper()
    stem = DATASET_STEM_TEMPLATE.format(ticker=symbol)
    return {
        "train": DATASET_BASE_DIR / f"{stem}_train.parquet",
        "val": DATASET_BASE_DIR / f"{stem}_val.parquet",
        "test": DATASET_BASE_DIR / f"{stem}_test.parquet",
        "dynamic_manifest_meta": DATASET_BASE_DIR / f"{stem}_dynamic_manifest_meta.json",
    }


def _load_split_parquet(parquet_path: Path, split_name: str, ticker_symbol: str) -> pd.DataFrame:
    if not parquet_path.exists():
        raise FileNotFoundError(f"[{ticker_symbol}] Missing split parquet: {parquet_path}")

    read_columns = sorted(REQUIRED_SPLIT_COLUMNS | {"entry_representation_raw_top5"})
    try:
        df_split = pd.read_parquet(parquet_path, columns=read_columns)
    except Exception:
        df_split = pd.read_parquet(parquet_path, columns=sorted(REQUIRED_SPLIT_COLUMNS))

    if "entry_representation" not in df_split.columns:
        if "entry_representation_raw_top5" in df_split.columns:
            df_split = df_split.copy()
            df_split["entry_representation"] = df_split["entry_representation_raw_top5"]
        else:
            raise ValueError(
                f"[{ticker_symbol}] Missing entry_representation column in {split_name} split. "
                "Expected entry_representation or entry_representation_raw_top5."
            )

    missing_cols = sorted(REQUIRED_SPLIT_COLUMNS - set(df_split.columns))
    if missing_cols:
        raise ValueError(
            f"[{ticker_symbol}] Missing required columns in {split_name} split: {missing_cols}"
        )

    df_split = df_split.copy()
    df_split["dataset_split"] = split_name
    return df_split


def _event_codes_from_df(df: pd.DataFrame, ticker_symbol: str) -> np.ndarray:
    if "event_type_competing" in df.columns:
        events = df["event_type_competing"].to_numpy(dtype=np.int64, copy=False)
    elif "event_type" in df.columns:
        events = df["event_type"].to_numpy(dtype=np.int64, copy=False)
    else:
        raise ValueError(f"[{ticker_symbol}] Could not find event_type or event_type_competing column.")
    return events


def _fit_feature_normalizer(x_train: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    feat_mean = x_train.mean(axis=(0, 1), keepdims=True).astype(np.float32)
    feat_std = x_train.std(axis=(0, 1), keepdims=True).astype(np.float32)
    feat_std = np.where(feat_std < 1e-6, 1.0, feat_std).astype(np.float32)

    side_idx = x_train.shape[2] - 2
    mask_idx = x_train.shape[2] - 1
    feat_mean[..., side_idx] = 0.0
    feat_std[..., side_idx] = 1.0
    feat_mean[..., mask_idx] = 0.0
    feat_std[..., mask_idx] = 1.0

    return feat_mean, feat_std


def _load_manifest_meta(meta_path: Path, ticker_symbol: str) -> tuple[dict, dict, float]:
    with open(meta_path, "r", encoding="utf-8") as f:
        manifest_meta = json.load(f)

    missing_keys = [k for k in REQUIRED_META_KEYS if k not in manifest_meta]
    if missing_keys:
        raise ValueError(f"[{ticker_symbol}] Missing keys in manifest metadata: {missing_keys}")

    t_max = float(manifest_meta["t_max"])
    if (not np.isfinite(t_max)) or t_max <= 0.0:
        raise ValueError(f"[{ticker_symbol}] Invalid t_max in metadata: {manifest_meta['t_max']!r}")

    preprocessing_config = manifest_meta.get("preprocessing_config")
    if not isinstance(preprocessing_config, dict):
        preprocessing_config = {}

    return manifest_meta, preprocessing_config, t_max


def _integrated_brier_score_to_tmax(bs_curve, time_grid) -> float:
    t_grid = np.asarray(time_grid, dtype=np.float64)
    bs_arr = np.asarray(bs_curve, dtype=np.float64)
    if t_grid.size == 0 or bs_arr.size == 0:
        return float("nan")

    t_min = float(t_grid[0])
    t_max_grid = float(t_grid[-1])
    span = t_max_grid - t_min
    if span <= 0.0:
        return float("nan")

    trapz_fn = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    return float(trapz_fn(bs_arr, t_grid) / span)


def safe_pr_auc_binary(y_true_binary, y_score):
    if int(y_true_binary.sum()) == 0:
        return float("nan"), np.array([1.0]), np.array([0.0])
    precision, recall, _ = precision_recall_curve(y_true_binary, y_score)
    pr_auc = float(auc(recall[::-1], precision[::-1]))
    return pr_auc, precision, recall


def safe_roc_auc_binary(y_true_binary, y_score):
    if np.unique(y_true_binary).size < 2:
        return float("nan")
    return float(roc_auc_score(y_true_binary, y_score))


def _set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _build_model(model_name_local: str, feature_dim: int, seq_len: int, output_steps: int) -> torch.nn.Module:
    if model_name_local == "gru":
        return DeepHitRNNCompeting(
            num_features=feature_dim,
            num_events=2,
            num_time_steps=output_steps,
            hidden_size=144,
            num_layers=2,
            rnn_dropout=0.2,
            fc_hidden=256,
            fc_dropout=0.2,
        ).to(device)
    if model_name_local == "gru_transformer":
        return DeepHitRNNTransformerCompeting(
            num_features=feature_dim,
            num_events=2,
            num_time_steps=output_steps,
            hidden_size=96,
            num_layers=2,
            rnn_dropout=0.2,
            transformer_layers=2,
            transformer_heads=4,
            transformer_ff_dim=192,
            transformer_dropout=0.1,
            max_seq_len=seq_len,
            fc_hidden=128,
            fc_dropout=0.2,
        ).to(device)
    if model_name_local == "transformer":
        return DeepHitTransformerCompeting(
            num_features=feature_dim,
            num_events=2,
            num_time_steps=output_steps,
            hidden_size=96,
            num_layers=2,
            num_heads=4,
            transformer_ff_dim=320,
            transformer_dropout=0.1,
            max_seq_len=seq_len,
            fc_hidden=112,
            fc_dropout=0.2,
        ).to(device)
    if model_name_local == "mamba":
        return DeepHitMambaCompeting(
            num_features=feature_dim,
            num_events=2,
            num_time_steps=output_steps,
            hidden_size=128,
            num_mamba_layers=2,
            d_state=16,
            d_conv=4,
            expand=2,
            mamba_dropout=0.15,
            fc_dropout=0.2,
        ).to(device)
    raise ValueError(f"Unknown model_name: {model_name_local}")


print(
    "Architecture note: keeping dynamic-grid architecture defaults for tuning parity. "
    "This differs from standardized_static_deephit baseline for GRU and Mamba hidden sizes."
)


TICKER_CONTEXT_CACHE: dict[str, dict] = {}


def _build_ticker_context(ticker_symbol: str) -> dict:
    symbol = str(ticker_symbol).upper()
    paths = _ticker_paths(symbol)

    missing = [str(p) for p in paths.values() if not p.exists()]
    if missing:
        raise FileNotFoundError(f"[{symbol}] Missing files:\n" + "\n".join(missing))

    df_train = _load_split_parquet(paths["train"], "train", symbol)
    df_val = _load_split_parquet(paths["val"], "val", symbol)
    _ = _load_split_parquet(paths["test"], "test", symbol)

    manifest_meta, preprocessing_config, t_max = _load_manifest_meta(
        paths["dynamic_manifest_meta"],
        symbol,
    )

    lookback_steps = int(preprocessing_config.get("lookback_steps", 500))
    lob_feature_dim = int(preprocessing_config.get("lob_feature_dim", 20))
    tox_feature_dim = int(preprocessing_config.get("tox_feature_dim", 12))
    artifact_num_time_steps = int(preprocessing_config.get("num_time_steps", NUM_TIME_STEPS))

    if artifact_num_time_steps != int(NUM_TIME_STEPS):
        raise ValueError(
            f"[{symbol}] num_time_steps mismatch: notebook={NUM_TIME_STEPS}, "
            f"artifact={artifact_num_time_steps}"
        )

    x_train_lob = extract_lob_features(df_train, lookback_steps, feat_dim=lob_feature_dim)
    x_train_tox = extract_toxicity_features(df_train, lookback_steps, feat_dim=tox_feature_dim)
    x_train = np.concatenate([x_train_lob, x_train_tox], axis=2)

    x_val_lob = extract_lob_features(df_val, lookback_steps, feat_dim=lob_feature_dim)
    x_val_tox = extract_toxicity_features(df_val, lookback_steps, feat_dim=tox_feature_dim)
    x_val = np.concatenate([x_val_lob, x_val_tox], axis=2)

    y_train = df_train["duration_s"].to_numpy(dtype=np.float32, copy=False)
    d_train = _event_codes_from_df(df_train, symbol)
    y_val = df_val["duration_s"].to_numpy(dtype=np.float32, copy=False)
    d_val = _event_codes_from_df(df_val, symbol)

    y_train_h, d_train_h, n_train_recensored = recensor_after_horizon(y_train, d_train, t_max)
    y_val_h, d_val_h, n_val_recensored = recensor_after_horizon(y_val, d_val, t_max)

    feat_mean, feat_std = _fit_feature_normalizer(x_train)
    x_train_norm = apply_dynamic_normalizer(x_train, feat_mean, feat_std)
    x_val_norm = apply_dynamic_normalizer(x_val, feat_mean, feat_std)

    label_transform = LabTransform(NUM_TIME_STEPS, scheme="quantiles")
    y_train_disc, d_train_disc = label_transform.fit_transform(y_train_h.copy(), d_train_h.copy())
    y_val_disc, d_val_disc = label_transform.transform(y_val_h.copy(), d_val_h.copy())

    time_grid = np.asarray(label_transform.cuts, dtype=np.float32)
    if time_grid.size == 0:
        raise ValueError(f"[{symbol}] Empty time_grid produced by label_transform.")

    if x_train_norm.shape[0] == 0 or x_val_norm.shape[0] == 0:
        raise ValueError(f"[{symbol}] Empty train/val split after preprocessing.")
    
    order_keys_train = np.arange(x_train_norm.shape[0], dtype=np.int64)
    
    order_keys_val = np.arange(x_val_norm.shape[0], dtype=np.int64)

    context = {
        "ticker": symbol,
        "manifest_meta": manifest_meta,
        "paths": paths,
        "X_train": x_train_norm,
        "X_val": x_val_norm,
        "train_sample_idx": np.arange(x_train_norm.shape[0], dtype=np.int64),
        "val_sample_idx": np.arange(x_val_norm.shape[0], dtype=np.int64),
        "Y_train": y_train_h.astype(np.float32, copy=False),
        "D_train": d_train_h.astype(np.int64, copy=False),
        "Y_val": y_val_h.astype(np.float32, copy=False),
        "D_val": d_val_h.astype(np.int64, copy=False),
        "Y_train_t": torch.tensor(y_train_disc, dtype=torch.int64, device="cpu"),
        "D_train_t": torch.tensor(d_train_disc, dtype=torch.int64, device="cpu"),
        "ORDER_KEYS_TRAIN": order_keys_train,
        "ORDER_KEYS_TRAIN_t": torch.tensor(order_keys_train, dtype=torch.int64, device="cpu"),
        "Y_val_t": torch.tensor(y_val_disc, dtype=torch.int64, device="cpu"),
        "D_val_t": torch.tensor(d_val_disc, dtype=torch.int64, device="cpu"),
        "ORDER_KEYS_VAL": order_keys_val,
        "feat_mean": feat_mean,
        "feat_std": feat_std,
        "time_grid": time_grid,
        "output_steps": int(time_grid.shape[0]),
        "feature_dim": int(x_train_norm.shape[2]),
        "seq_len": int(x_train_norm.shape[1]),
        "aux_target_dim": int(x_train_norm.shape[2] - 2),
        "t_max": float(t_max),
        "n_train_recensored": int(n_train_recensored),
        "n_val_recensored": int(n_val_recensored),
    }

    print(
        f"[{symbol}] loaded train={x_train_norm.shape[0]:,}, val={x_val_norm.shape[0]:,}, "
        f"feature_dim={context['feature_dim']}, seq_len={context['seq_len']}, "
        f"output_steps={context['output_steps']}, t_max={context['t_max']:.6g}, "
        f"recensored(train/val)=({context['n_train_recensored']}/{context['n_val_recensored']})"
    )
    return context


def get_ticker_context(ticker_symbol: str) -> dict:
    symbol = str(ticker_symbol).upper()
    if symbol not in TICKER_CONTEXT_CACHE:
        TICKER_CONTEXT_CACHE[symbol] = _build_ticker_context(symbol)
    return TICKER_CONTEXT_CACHE[symbol]


def _iter_train_batches(context: dict, orders_per_batch: int, seed: int):
    batch_indices = build_order_batch_indices(
        context["ORDER_KEYS_TRAIN"],
        orders_per_batch=orders_per_batch,
        shuffle=True,
        seed=seed,
    )
    for idx_np in batch_indices:
        idx_t = torch.as_tensor(idx_np, dtype=torch.int64, device="cpu")
        x_b_cpu = torch.from_numpy(context["X_train"][idx_np])

        yield (
            x_b_cpu,
            context["Y_train_t"].index_select(0, idx_t),
            context["D_train_t"].index_select(0, idx_t),
            context["ORDER_KEYS_TRAIN_t"].index_select(0, idx_t),
        )


def _static_ctd_event(
    cif_event: np.ndarray,
    durations: np.ndarray,
    events: np.ndarray,
    event_code: int,
    time_grid_local: np.ndarray,
    eps: float = 1e-12,
) -> float:
    n = len(durations)
    if n == 0:
        return float("nan")

    tau_idx = np.searchsorted(time_grid_local, durations, side="left")
    tau_idx = np.clip(tau_idx, 0, len(time_grid_local) - 1)

    num = 0.0
    den = 0.0

    anchors = np.where(events == int(event_code))[0]
    for i in anchors:
        later_idx = np.where(durations > durations[i])[0]
        if later_idx.size == 0:
            continue

        tau_anchor = int(tau_idx[i])
        s_i = float(cif_event[tau_anchor, i])
        s_j = cif_event[tau_anchor, later_idx]

        concordant = (s_i > s_j).astype(np.float64)
        ties = (s_i == s_j).astype(np.float64)
        num += float(np.sum(concordant + 0.5 * ties))
        den += float(later_idx.size)

    if den <= eps:
        return float("nan")
    return num / den


def compute_validation_metrics(base_net: torch.nn.Module, context: dict) -> dict:
    base_net.eval()
    n_val = int(context["X_val"].shape[0])
    output_steps = int(context["output_steps"])
    cif_val = np.empty((len(EVENT_CODES), output_steps, n_val), dtype=np.float32)

    with torch.no_grad():
        for start in range(0, n_val, BATCH_SIZE):
            x_b_np = context["X_val"][start : start + BATCH_SIZE]
            x_b = torch.from_numpy(x_b_np).to(device, non_blocking=LOADER_PIN_MEMORY)
            logits_b = base_net(x_b)
            pmf_b = torch.softmax(logits_b.reshape(logits_b.size(0), -1), dim=1).reshape(
                logits_b.size(0),
                len(EVENT_CODES),
                output_steps,
            )
            cif_b = torch.cumsum(pmf_b, dim=2).cpu().numpy()
            end = start + cif_b.shape[0]
            cif_val[:, :, start:end] = np.transpose(cif_b, (1, 2, 0))

    metrics = {}

    ctd_scores = {}
    for k, (event_code, event_name) in enumerate(zip(EVENT_CODES, EVENT_NAMES)):
        ctd_scores[event_name] = _static_ctd_event(
            cif_event=cif_val[k],
            durations=context["Y_val"],
            events=context["D_val"],
            event_code=int(event_code),
            time_grid_local=context["time_grid"],
        )

    ctd_values = np.asarray([ctd_scores.get(name, np.nan) for name in EVENT_NAMES], dtype=np.float64)
    ctd_macro = float(np.nanmean(ctd_values)) if not np.all(np.isnan(ctd_values)) else float("nan")

    metrics["val_ctd_macro"] = ctd_macro
    metrics["val_ctd_favorable"] = float(ctd_scores.get("FAVORABLE_FILL", np.nan))
    metrics["val_ctd_toxic"] = float(ctd_scores.get("TOXIC_FILL", np.nan))

    p_fav_train_curve = uninformed_cif_curve_from_train(
        context["Y_train"],
        context["D_train"],
        1,
        context["time_grid"],
    )
    p_tox_train_curve = uninformed_cif_curve_from_train(
        context["Y_train"],
        context["D_train"],
        2,
        context["time_grid"],
    )

    bs_curves = {}
    for k, (event_code, event_name) in enumerate(zip(EVENT_CODES, EVENT_NAMES)):
        bs_curves[event_name] = standard_brier_score(
            durations=context["Y_val"],
            events=context["D_val"],
            cif_k=cif_val[k],
            event_code=event_code,
            time_grid=context["time_grid"],
        )

    bs_curves_uninformed = {
        "FAVORABLE_FILL": uninformed_brier_score(
            context["Y_val"],
            context["D_val"],
            p_fav_train_curve,
            1,
            context["time_grid"],
        ),
        "TOXIC_FILL": uninformed_brier_score(
            context["Y_val"],
            context["D_val"],
            p_tox_train_curve,
            2,
            context["time_grid"],
        ),
    }

    ibs_scores = {}
    ibs_scores_uninformed = {}
    for event_name, bs_arr in bs_curves.items():
        ibs_scores[event_name] = _integrated_brier_score_to_tmax(
            bs_arr,
            context["time_grid"],
        )
        ibs_scores_uninformed[event_name] = _integrated_brier_score_to_tmax(
            bs_curves_uninformed[event_name],
            context["time_grid"],
        )

    ibs_vals = np.asarray([ibs_scores.get(name, np.nan) for name in EVENT_NAMES], dtype=np.float64)
    ibs_vals_u = np.asarray([ibs_scores_uninformed.get(name, np.nan) for name in EVENT_NAMES], dtype=np.float64)
    ibs_macro = float(np.nanmean(ibs_vals)) if not np.all(np.isnan(ibs_vals)) else float("nan")
    ibs_macro_uninf = float(np.nanmean(ibs_vals_u)) if not np.all(np.isnan(ibs_vals_u)) else float("nan")

    metrics["val_ibs_macro"] = ibs_macro
    metrics["val_ibs_macro_uninformed"] = ibs_macro_uninf
    metrics["val_ibs_favorable"] = float(ibs_scores.get("FAVORABLE_FILL", np.nan))
    metrics["val_ibs_toxic"] = float(ibs_scores.get("TOXIC_FILL", np.nan))

    final_cif_tox = cif_val[1, -1, :]
    y_toxic = (context["D_val"] == 2).astype(int)
    ap_toxic, _, _ = safe_pr_auc_binary(y_toxic, final_cif_tox)
    auc_toxic = safe_roc_auc_binary(y_toxic, final_cif_tox)

    metrics["val_toxic_pr_auc"] = float(ap_toxic)
    metrics["val_toxic_roc_auc"] = float(auc_toxic)

    # Compatibility aliases so existing aggregation tooling keeps working.
    metrics["val_weighted_ctd"] = float(metrics["val_ctd_macro"])
    metrics["val_ibs_weighted"] = float(metrics["val_ibs_macro"])
    metrics["val_ibs_weighted_uninformed"] = float(metrics["val_ibs_macro_uninformed"])
    metrics["val_toxic_pr_auc_weighted"] = float(metrics["val_toxic_pr_auc"])
    metrics["val_toxic_pr_auc_sample"] = float(metrics["val_toxic_pr_auc"])
    metrics["val_toxic_roc_auc_weighted"] = float(metrics["val_toxic_roc_auc"])
    metrics["val_toxic_roc_auc_sample"] = float(metrics["val_toxic_roc_auc"])

    return metrics


def run_single_experiment(context: dict, alpha: float, beta_l3: float, seed: int) -> dict:
    ticker = context["ticker"]
    _set_seed(seed)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    base_net = _build_model(
        model_name_local=str(model_name).lower(),
        feature_dim=int(context["feature_dim"]),
        seq_len=int(context["seq_len"]),
        output_steps=int(context["output_steps"]),
    )

    optimizer = torch.optim.Adam(base_net.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,
        eta_min=LEARNING_RATE * 0.1,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    run_name = (
        f"{ticker}_seed{seed}_alpha{alpha:.6g}_beta{beta_l3:.6g}_{str(model_name).lower()}"
    )

    wandb_run = None
    if WANDB_ENABLE:
        init_kwargs = {
            "project": WANDB_PROJECT,
            "name": run_name,
            "config": {
                "ticker": ticker,
                "seed": int(seed),
                "model_name": str(model_name).lower(),
                "alpha": float(alpha),
                "beta_l3": float(beta_l3),
                "sigma": float(SIGMA),
                "learning_rate": float(LEARNING_RATE),
                "num_epochs": int(num_epochs),
                "orders_per_batch": int(ORDERS_PER_BATCH),
                "batch_size_eval": int(BATCH_SIZE),
                "t_max": float(context["t_max"]),
            },
            "mode": WANDB_MODE,
            "reinit": True,
        }
        if WANDB_ENTITY:
            init_kwargs["entity"] = WANDB_ENTITY
        wandb_run = wandb.init(**init_kwargs)

    for epoch in range(num_epochs):
        base_net.train()
        epoch_total = 0.0
        epoch_l1 = 0.0
        epoch_l2 = 0.0
        epoch_l3 = 0.0
        n_batches = 0

        for X_b_cpu, Y_b_cpu, D_b_cpu, O_b_cpu in _iter_train_batches(
            context,
            ORDERS_PER_BATCH,
            seed + epoch,
        ):
            X_b = X_b_cpu.to(device, non_blocking=LOADER_PIN_MEMORY)
            Y_b = Y_b_cpu.to(device, non_blocking=LOADER_PIN_MEMORY)
            D_b = D_b_cpu.to(device, non_blocking=LOADER_PIN_MEMORY)
            O_b = O_b_cpu.to(device, non_blocking=LOADER_PIN_MEMORY)

            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=AUTOCAST_DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
                logits = base_net(X_b)
                total_loss, parts = static_deephit_total_loss(
                    logits=logits,
                    y=Y_b,
                    d=D_b,
                    order_ids=O_b,
                    x_batch=X_b,
                    base_net=base_net,
                    alpha=float(alpha),
                    sigma=float(SIGMA),
                    beta_l3=float(beta_l3),
                    aux_target_dim=int(context["aux_target_dim"]),
                )

            if USE_AMP:
                scaler.scale(total_loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                total_loss.backward()
                optimizer.step()

            epoch_total += float(total_loss.item())
            epoch_l1 += float(parts["l1"].item())
            epoch_l2 += float(parts["l2"].item())
            epoch_l3 += float(parts["l3"].item())
            n_batches += 1

        if n_batches == 0:
            raise RuntimeError(f"[{ticker}] No training batches were produced.")

        scheduler.step()
        train_total = epoch_total / n_batches
        train_l1 = epoch_l1 / n_batches
        train_l2 = epoch_l2 / n_batches
        train_l3 = epoch_l3 / n_batches
        lr_now = float(optimizer.param_groups[0]["lr"])

        if wandb_run is not None:
            wandb_run.log(
                {
                    "epoch": epoch + 1,
                    "train/total_loss": train_total,
                    "train/l1_loss": train_l1,
                    "train/l2_loss": train_l2,
                    "train/l3_loss": train_l3,
                    "train/lr": lr_now,
                },
                step=epoch + 1,
            )

    metrics = compute_validation_metrics(base_net, context)

    result = {
        "ticker": ticker,
        "seed": int(seed),
        "alpha": float(alpha),
        "beta_l3": float(beta_l3),
        "model_name": str(model_name).lower(),
        "num_epochs": int(num_epochs),
        "learning_rate": float(LEARNING_RATE),
        "run_name": run_name,
        **metrics,
    }

    if wandb_run is not None:
        wandb_eval = {f"eval/{k}": v for k, v in metrics.items()}
        wandb_run.log(wandb_eval)
        for k, v in metrics.items():
            wandb_run.summary[k] = v
        wandb_run.finish()

    return result

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.


wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /jet/home/ccheung1/.netrc


Device: cuda
PyTorch: 2.6.0+cu124
Python: /ocean/projects/cis260122p/ccheung1/.conda_envs/lob/bin/python3.11
causal_conv1d: 1.6.1
mamba_ssm: 2.3.1
Architecture note: keeping dynamic-grid architecture defaults for tuning parity. This differs from standardized_static_deephit baseline for GRU and Mamba hidden sizes.


In [ ]:
TICKERS = [str(t).upper() for t in TICKERS]
if not TICKERS:
    raise ValueError("TICKERS must contain at least one symbol.")

for ticker_symbol in TICKERS:
    _ = get_ticker_context(ticker_symbol)

DEFAULT_ALPHA_GRID = [0.5, 1.0, 2.0, 4.0, 8.0, 12.0, 16.0]
DEFAULT_BETA_GRID = [0, 0.05, 0.1, 0.2, 0.4]


alpha_grid = DEFAULT_ALPHA_GRID
beta_grid = DEFAULT_BETA_GRID

if selected_alpha is not None:
    alpha_grid = [float(selected_alpha)]
if selected_beta_l3 is not None:
    beta_grid = [float(selected_beta_l3)]

print("Tickers:", TICKERS)
print("Seeds:", SEEDS)
print("Alpha grid:", alpha_grid)
print("Beta grid:", beta_grid)
if selected_alpha is not None or selected_beta_l3 is not None:
    print("Shard mode enabled.")
else:
    print("Full grid mode enabled.")

results: list[dict] = []
total_runs = len(alpha_grid) * len(beta_grid) * len(TICKERS) * len(SEEDS)
run_idx = 0

for alpha in alpha_grid:
    for beta_l3 in beta_grid:
        print(f"\nConfig alpha={alpha:.6g}, beta={beta_l3:.6g}")
        for ticker_symbol in TICKERS:
            ticker_context = get_ticker_context(ticker_symbol)
            for seed in SEEDS:
                run_idx += 1
                print(
                    f"[{run_idx}/{total_runs}] ticker={ticker_symbol}, seed={seed}, alpha={alpha:.6g}, beta={beta_l3:.6g}, model={str(model_name).lower()}"
                )
                run_result = run_single_experiment(
                    context=ticker_context,
                    alpha=float(alpha),
                    beta_l3=float(beta_l3),
                    seed=int(seed),
                )
                if str(array_job_id).strip():
                    run_result["array_job_id"] = str(array_job_id).strip()
                    run_result["array_task_id"] = int(array_task_id)
                results.append(run_result)

GRID_RESULTS_DF = pd.DataFrame(results)
if GRID_RESULTS_DF.empty:
    raise RuntimeError("No runs were completed.")

ctd_col = "val_ctd_macro" if "val_ctd_macro" in GRID_RESULTS_DF.columns else "val_weighted_ctd"
ibs_col = "val_ibs_macro" if "val_ibs_macro" in GRID_RESULTS_DF.columns else "val_ibs_weighted"
pr_col = "val_toxic_pr_auc" if "val_toxic_pr_auc" in GRID_RESULTS_DF.columns else "val_toxic_pr_auc_weighted"

GRID_RESULTS_DF = GRID_RESULTS_DF.sort_values(ctd_col, ascending=False).reset_index(drop=True)

agg_cols = {
    ctd_col: ["mean", "std"],
    ibs_col: ["mean"],
    pr_col: ["mean"],
}

PER_TICKER_AGG_DF = (
    GRID_RESULTS_DF
    .groupby(["ticker", "alpha", "beta_l3", "model_name"], as_index=False)
    .agg(
        **{
            f"mean_{col}": (col, "mean") for col in agg_cols
        },
        **{
            f"std_{col}": (col, "std") for col, funcs in agg_cols.items() if "std" in set(funcs)
        },
        runs=(ctd_col, "count")
    )
)

BEST_PER_TICKER_DF = (
    PER_TICKER_AGG_DF
    .loc[PER_TICKER_AGG_DF.groupby("ticker")[f"mean_{ctd_col}"].idxmax()]
    .sort_values(["ticker", f"mean_{ctd_col}"], ascending=[True, False])
    .reset_index(drop=True)
)

GLOBAL_AGG_DF = (
    GRID_RESULTS_DF
    .groupby(["alpha", "beta_l3", "model_name"], as_index=False)
    .agg(
        **{
            f"mean_{col}": (col, "mean") for col in agg_cols
        },
        **{
            f"std_{col}": (col, "std") for col, funcs in agg_cols.items() if "std" in set(funcs)
        },
        runs=(ctd_col, "count")
    )
    .sort_values(f"mean_{ctd_col}", ascending=False)
    .reset_index(drop=True)
)
BEST_GLOBAL_DF = GLOBAL_AGG_DF.head(1).copy()

print("\nTop individual runs by validation C-td")
display(GRID_RESULTS_DF.head(20))

print("\nBest alpha/beta per ticker (aggregated across seeds)")
display(BEST_PER_TICKER_DF)

print("\nSingle global best alpha/beta (aggregated across all tickers and seeds)")
display(BEST_GLOBAL_DF)

# Persist results from this execution shard so array tasks can be aggregated later.
def _safe_value_tag(value: float) -> str:
    return f"{float(value):.6g}".replace("-", "m").replace(".", "p")

alpha_tag = _safe_value_tag(alpha_grid[0]) if len(alpha_grid) == 1 else "multi"
beta_tag = _safe_value_tag(beta_grid[0]) if len(beta_grid) == 1 else "multi"
model_tag = str(model_name).lower().replace("/", "_")
array_job_id_str = str(array_job_id).strip()
array_task_id_str = str(array_task_id).strip() if array_task_id is not None else ""
shard_label = f"{array_job_id_str}_{array_task_id_str}" if array_job_id_str else "local"

results_root = (repo_root if repo_root is not None else Path.cwd()) / "results"
shard_dir = results_root / "grid_search_shards"
shard_dir.mkdir(parents=True, exist_ok=True)

shard_stem = f"grid_results_{model_tag}_{shard_label}_a{alpha_tag}_b{beta_tag}"
shard_csv_path = shard_dir / f"{shard_stem}.csv"
shard_meta_path = shard_dir / f"{shard_stem}.json"

GRID_RESULTS_DF.to_csv(shard_csv_path, index=False)

shard_meta = {
    "array_job_id": array_job_id_str,
    "array_task_id": array_task_id_str,
    "model_name": model_tag,
    "alpha_grid": [float(a) for a in alpha_grid],
    "beta_grid": [float(b) for b in beta_grid],
    "tickers": [str(t) for t in TICKERS],
    "seeds": [int(s) for s in SEEDS],
    "rows": int(len(GRID_RESULTS_DF)),
    "csv_path": str(shard_csv_path),
    "created_utc": pd.Timestamp.now(tz="UTC").isoformat(),
}
with open(shard_meta_path, "w", encoding="utf-8") as f:
    json.dump(shard_meta, f, indent=2)

print(f"\nSaved shard results: {shard_csv_path}")
print(f"Saved shard metadata: {shard_meta_path}")

[AAPL] loaded train=43,000, val=10,000, feature_dim=34, seq_len=500, output_steps=30, t_max=44.4995, recensored(train/val)=(2150/503)
Tickers: ['AAPL']
Seeds: [4718]
Alpha grid: [1.0]
Beta grid: [0.0]
Full grid mode enabled.

Config alpha=1, beta=0
[1/1] ticker=AAPL, seed=4718, alpha=1, beta=0, model=mamba


epoch,▁▃▅▆█
eval/val_ctd_favorable,▁
eval/val_ctd_macro,▁
eval/val_ctd_toxic,▁
eval/val_ibs_favorable,▁
eval/val_ibs_macro,▁
eval/val_ibs_macro_uninformed,▁
eval/val_ibs_toxic,▁
eval/val_ibs_weighted,▁
eval/val_ibs_weighted_uninformed,▁
+12,...



Top individual runs by validation C-td


,ticker,seed,alpha,beta_l3,model_name,num_epochs,learning_rate,run_name,val_ctd_macro,val_ctd_favorable,...,val_ibs_toxic,val_toxic_pr_auc,val_toxic_roc_auc,val_weighted_ctd,val_ibs_weighted,val_ibs_weighted_uninformed,val_toxic_pr_auc_weighted,val_toxic_pr_auc_sample,val_toxic_roc_auc_weighted,val_toxic_roc_auc_sample
0,AAPL,4718,1.0,0.0,mamba,5,0.0005,AAPL_seed4718_alpha1_beta0_mamba,0.652178,0.633621,...,0.190365,0.277575,0.611134,0.652178,0.199394,0.170936,0.277575,0.277575,0.611134,0.611134



Best alpha/beta per ticker (aggregated across seeds)


,ticker,alpha,beta_l3,model_name,mean_val_ctd_macro,mean_val_ibs_macro,mean_val_toxic_pr_auc,std_val_ctd_macro,runs
0,AAPL,1.0,0.0,mamba,0.652178,0.199394,0.277575,NaN,1



Single global best alpha/beta (aggregated across all tickers and seeds)


,alpha,beta_l3,model_name,mean_val_ctd_macro,mean_val_ibs_macro,mean_val_toxic_pr_auc,std_val_ctd_macro,runs
0,1.0,0.0,mamba,0.652178,0.199394,0.277575,NaN,1



Saved shard results: /ocean/projects/cis260122p/ccheung1/lob-deep-survival-analysis/results/grid_search_shards/grid_results_mamba_local_a1_b0.csv
Saved shard metadata: /ocean/projects/cis260122p/ccheung1/lob-deep-survival-analysis/results/grid_search_shards/grid_results_mamba_local_a1_b0.json
